In [1]:
"""
=============================================================================
ABLACAO DE SCHEDULERS COM HPO POR CONFIGURACAO - ResNet50 / Displasia Epitelial
Oral (4 classes)
=============================================================================

Objetivo: DIFERENTE do notebook pipeline_resnet18.ipynb (que fixa LR / weight
decay / batch size para isolar apenas o EFEITO DO SCHEDULER), aqui cada
combinacao (scheduler, seed) recebe sua PROPRIA otimizacao Optuna antes do
treino completo. A pergunta que este notebook responde e':

    "qual e' o melhor desempenho que cada scheduler consegue alcancar,
     dado que cada um recebe seu proprio HPO?"

isto e', NAO controla mais o fator "qualidade do HPO" -- ele agora faz parte
do que esta sendo comparado. Para a comparacao pareada "tudo igual, so o
scheduler muda", use pipeline_resnet18.ipynb.

Fluxo por combinacao (scheduler, seed):
    1. prepare_split(seed)      -> mesmo holdout treino/CV vs. teste do notebook
                                    original (garante que HPO nunca toca o teste)
    2. optimize_config(...)     -> Optuna (TPE + Hyperband) busca lr / weight_decay
                                    / batch_size / hiperparametros especificos do
                                    scheduler, com orcamento REDUZIDO
                                    (HPO_N_TRIALS x HPO_TRIAL_EPOCHS x HPO_FOLDS)
    3. run_config(...)          -> treino completo (N_FOLDS x EPOCHS) com os
                                    hiperparametros vencedores do passo 2

Custo: isto ADICIONA um HPO completo antes de cada um dos len(SCHEDULERS) x
len(SEEDS) treinos completos. Com os defaults abaixo (15 trials x 12 epocas x
1 fold) o HPO ja aproximadamente DOBRA o tempo total da ablacao. Ajuste
HPO_N_TRIALS / HPO_TRIAL_EPOCHS / HPO_FOLDS para o seu orcamento de GPU.

Uso:
    python pipeline_resnet50_optuna.py
    python pipeline_resnet50_optuna.py --summary [caminho.csv]
=============================================================================
"""

import os
import csv
import random
from datetime import datetime
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import optuna
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    confusion_matrix, f1_score, roc_auc_score, accuracy_score, cohen_kappa_score
)
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader, Subset
from torch.optim.lr_scheduler import OneCycleLR, CosineAnnealingLR, ReduceLROnPlateau
from torchvision import datasets, models, transforms
import warnings
import numpy as np

try:
    import matplotlib.pyplot as plt
    import matplotlib.ticker as mticker
    PLOTTING_AVAILABLE = True
except ImportError:
    plt = None
    mticker = None
    PLOTTING_AVAILABLE = False

optuna.logging.set_verbosity(optuna.logging.WARNING)

c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# =============================================================================
# 1. CONFIGURACAO
# =============================================================================
SCHEDULERS = ["CosineAnnealingLR", "OneCycleLR", "ReduceLROnPlateau"]
SEEDS      = [42, 17, 7, 12, 123, 555, 996]

# --- Hiperparametros PADRAO: usados so como fallback (nunca deveriam ser
# usados na pratica, ja que main() sempre passa os valores vindos do Optuna) ---
LR           = 1e-3
WEIGHT_DECAY = 1e-5
BATCH_SIZE   = 32
EPOCHS       = 50
N_FOLDS      = 5
TEST_SIZE    = 0.2
MODEL_NAME   = "ResNet50_Weights.DEFAULT"

CHECKPOINT_METRIC = "clinical_composite"  # opcoes: "auc", "f1", "qwk", "clinical_composite"
DANGER_RATE_WEIGHT = 1.0
UNDEREST_RATE_WEIGHT = 1.0
SAVE_TELEMETRY = True

NUM_WORKERS = 0  # ver nota em pipeline_resnet18.ipynb sobre Windows + DataLoader workers

BASE_DIR   = "Originais"
OUTPUT_DIR = f"./ablation_schedulers_resnet50_optuna_ckpt_{CHECKPOINT_METRIC}"
CSV_FILE   = os.path.join(OUTPUT_DIR, "ablation_logs_resnet50_optuna.csv")
HPO_TRIALS_DIR = os.path.join(OUTPUT_DIR, "optuna_trials")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(HPO_TRIALS_DIR, exist_ok=True)

ORDINAL_ORDER = ["healthy", "mild", "moderate", "severe"]

DETERMINISTIC = True

# --- Orcamento do HPO (por combinacao scheduler x seed) ---
HPO_N_TRIALS     = 15     # numero de trials do Optuna por (scheduler, seed)
HPO_TRIAL_EPOCHS = 12     # epocas de treino DENTRO de cada trial (< EPOCHS da run final)
HPO_FOLDS        = 1      # folds usados so' para avaliar cada trial (1 = split unico, rapido)

# Espacos de busca (lr / weight_decay / batch_size sao comuns a todos os schedulers)
HPO_LR_RANGE = (1e-5, 5e-3)          # log-uniform
HPO_WD_RANGE = (1e-6, 1e-2)          # log-uniform
HPO_BATCH_SIZES = [8, 16, 32]

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Rodando em: {device}")

Rodando em: cuda


In [4]:
def seed_everything(seed: int):
    """Fixa todas as fontes de aleatoriedade para reprodutibilidade."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if DETERMINISTIC:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True

In [5]:
DATA_TRANSFORMS = {
    "treino": transforms.Compose([
        transforms.Resize(256),
        transforms.RandomCrop(224),
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomVerticalFlip(0.5),
        transforms.RandomRotation(40),
        transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]),
    "teste": transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]),
}


def make_loader_generator(seed: int):
    g = torch.Generator()
    g.manual_seed(seed)
    return g


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def build_model(num_classes: int):
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def make_scheduler(name: str, optimizer, steps_per_epoch: int, epochs: int, **kwargs):
    """
    Retorna (scheduler, step_mode). kwargs permite que o Optuna sobrescreva
    hiperparametros especificos do scheduler (ex.: pct_start do OneCycleLR);
    quando vazio, usa os mesmos defaults de pipeline_resnet18.ipynb.
    step_mode:
      'batch'   -> scheduler.step() apos cada batch          (OneCycleLR)
      'epoch'   -> scheduler.step() ao fim da epoca          (CosineAnnealingLR)
      'plateau' -> scheduler.step(val_loss) ao fim da epoca  (ReduceLROnPlateau)
    """
    if name == "OneCycleLR":
        params = dict(max_lr=optimizer.param_groups[0]["lr"],
                      steps_per_epoch=steps_per_epoch, epochs=epochs)
        params.update(kwargs)
        sch = OneCycleLR(optimizer, **params)
        return sch, "batch"
    if name == "CosineAnnealingLR":
        params = dict(T_max=epochs, eta_min=1e-8)
        params.update(kwargs)
        sch = CosineAnnealingLR(optimizer, **params)
        return sch, "epoch"
    if name == "ReduceLROnPlateau":
        params = dict(mode="min", factor=0.5, patience=5, min_lr=1e-8)
        params.update(kwargs)
        sch = ReduceLROnPlateau(optimizer, **params)
        return sch, "plateau"
    raise ValueError(f"Scheduler desconhecido: {name}")


def suggest_scheduler_kwargs(trial, scheduler_name: str) -> dict:
    """Espaco de busca especifico de cada scheduler (alem de lr/wd/batch_size)."""
    if scheduler_name == "OneCycleLR":
        return {"pct_start": trial.suggest_float("pct_start", 0.1, 0.4)}
    if scheduler_name == "CosineAnnealingLR":
        return {"eta_min": trial.suggest_float("eta_min", 1e-9, 1e-6, log=True)}
    if scheduler_name == "ReduceLROnPlateau":
        return {
            "factor": trial.suggest_float("factor", 0.1, 0.7),
            "patience": trial.suggest_int("patience", 2, 8),
        }
    raise ValueError(f"Scheduler desconhecido: {scheduler_name}")


def ordinal_ranks(class_names):
    """Mapeia o indice de classe do ImageFolder -> rank ordinal clinico (0..3)."""
    missing = [c for c in class_names if c not in ORDINAL_ORDER]
    if missing:
        raise ValueError(
            f"Classes sem rank ordinal: {missing}. Ajuste ORDINAL_ORDER={ORDINAL_ORDER}."
        )
    return np.array([ORDINAL_ORDER.index(c) for c in class_names])

In [6]:
def clinical_metrics(y_true, y_pred, rank_lookup):
    """
    QWK (quadratic), acuracia, erros perigosos e subestimacao clinica.
    Erro perigoso  = distancia ordinal >= 2 (em qualquer direcao).
    Subestimacao   = prever classe MENOS grave que a real, com distancia >= 2.
    """
    yt = rank_lookup[np.asarray(y_true)]
    yp = rank_lookup[np.asarray(y_pred)]
    n = len(yt)

    qwk = cohen_kappa_score(yt, yp, weights="quadratic")
    acc = accuracy_score(yt, yp)

    dist = np.abs(yt - yp)
    danger = int((dist >= 2).sum())
    under = int(((yp < yt) & (dist >= 2)).sum())
    return {
        "qwk": qwk,
        "acc": acc,
        "danger_count": danger,
        "danger_rate": danger / n,
        "underest_count": under,
        "underest_rate": under / n,
    }


def composite_score(clin: dict) -> float:
    """Mesmo score usado para selecao de checkpoint E como objetivo do Optuna."""
    return (clin["qwk"]
            - DANGER_RATE_WEIGHT * clin["danger_rate"]
            - UNDEREST_RATE_WEIGHT * clin["underest_rate"])


def checkpoint_selection_score(val_auc, val_f1, y_true, y_pred, rank_lookup):
    """Score usado para escolher o checkpoint salvo em cada fold."""
    clin = clinical_metrics(y_true, y_pred, rank_lookup)
    metric = CHECKPOINT_METRIC.lower()
    if metric == "auc":
        score = val_auc if not np.isnan(val_auc) else -np.inf
    elif metric == "f1":
        score = val_f1
    elif metric == "qwk":
        score = clin["qwk"]
    elif metric == "clinical_composite":
        score = composite_score(clin)
    else:
        raise ValueError(
            "CHECKPOINT_METRIC deve ser: 'auc', 'f1', 'qwk' ou 'clinical_composite'."
        )
    if np.isnan(score):
        score = -np.inf
    return score, clin

In [7]:
def plot_fold_telemetry(history: dict, fold: int, scheduler_name: str, seed: int,
                        save_dir: str = OUTPUT_DIR):
    """
    Gera um painel com 3 graficos de linha para auditar o treinamento do fold.
    history deve conter: 'train_loss', 'val_loss', 'train_acc', 'val_acc', 'lrs'
    """
    if not SAVE_TELEMETRY:
        return None
    if not PLOTTING_AVAILABLE:
        warnings.warn("matplotlib indisponivel; telemetria visual nao sera salva.")
        return None

    epochs = range(1, len(history['train_loss']) + 1)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5), dpi=120)
    fig.suptitle(f"Auditoria de Treinamento - {scheduler_name} | Seed {seed} | Fold {fold}",
                 fontsize=16, fontweight='bold')

    axes[0].plot(epochs, history['train_loss'], label='Train Loss', color='#1f77b4', linewidth=2)
    axes[0].plot(epochs, history['val_loss'], label='Validation Loss', color='#ff7f0e', linewidth=2)
    axes[0].set_title("Evolucao da Funcao de Custo (CrossEntropy)")
    axes[0].set_xlabel("Epocas")
    axes[0].set_ylabel("Loss")
    axes[0].grid(True, linestyle='--', alpha=0.6)
    axes[0].legend()

    axes[1].plot(epochs, history['train_acc'], label='Train Accuracy', color='#1f77b4', linewidth=2)
    axes[1].plot(epochs, history['val_acc'], label='Validation Accuracy', color='#2ca02c', linewidth=2)
    axes[1].set_title("Evolucao da Acuracia")
    axes[1].set_xlabel("Epocas")
    axes[1].set_ylabel("Acuracia")
    axes[1].set_ylim([0.0, 1.05])
    axes[1].grid(True, linestyle='--', alpha=0.6)
    axes[1].legend()

    axes[2].plot(epochs, history['lrs'], color='#9467bd', linewidth=2)
    axes[2].set_title(f"Evolucao do Learning Rate ({scheduler_name})")
    axes[2].set_xlabel("Epocas")
    axes[2].set_ylabel("Learning Rate")
    axes[2].grid(True, linestyle='--', alpha=0.6)
    axes[2].yaxis.set_major_locator(mticker.MaxNLocator(nbins=8))
    axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.1e}"))

    plt.tight_layout()

    file_path = os.path.join(save_dir, f"telemetry_{scheduler_name}_seed{seed}_fold{fold}.png")
    plt.savefig(file_path)
    print(f" [Grafico] Telemetria salva em: {file_path}")
    plt.close()
    return file_path

In [8]:
def prepare_split(seed: int) -> dict:
    """
    Prepara datasets + split holdout (treino/CV vs. teste) para uma seed.
    Usado tanto pelo Optuna (que so' enxerga cv_idx, nunca test_idx) quanto
    pelo treino final -> garante que os dois usam EXATAMENTE a mesma particao.
    """
    seed_everything(seed)

    if not os.path.isdir(BASE_DIR):
        raise FileNotFoundError(f"BASE_DIR nao encontrado: {os.path.abspath(BASE_DIR)}")

    full_aug   = datasets.ImageFolder(root=BASE_DIR, transform=DATA_TRANSFORMS["treino"])
    full_plain = datasets.ImageFolder(root=BASE_DIR, transform=DATA_TRANSFORMS["teste"])
    num_classes = len(full_aug.classes)
    class_names = full_aug.classes
    rank_lookup = ordinal_ranks(class_names)
    all_labels  = np.array(full_aug.targets)

    cv_idx, test_idx = train_test_split(
        np.arange(len(all_labels)), test_size=TEST_SIZE,
        stratify=all_labels, random_state=seed,
    )
    train_labels = all_labels[cv_idx]

    return {
        "full_aug": full_aug, "full_plain": full_plain,
        "num_classes": num_classes, "class_names": class_names,
        "rank_lookup": rank_lookup, "all_labels": all_labels,
        "cv_idx": cv_idx, "test_idx": test_idx, "train_labels": train_labels,
    }

In [9]:
def hpo_objective(trial, scheduler_name: str, seed: int, split: dict) -> float:
    """
    Treina uma versao BARATA (HPO_TRIAL_EPOCHS epocas, HPO_FOLDS fold(s)) da
    config sugerida pelo trial, usando SOMENTE o pool treino/CV (split["cv_idx"])
    -- o holdout de teste (split["test_idx"]) nunca e' tocado aqui.
    Retorna o mesmo score composto (QWK - danger_rate - underest_rate) usado
    para selecao de checkpoint no treino final, para manter os dois alinhados.
    """
    lr = trial.suggest_float("lr", *HPO_LR_RANGE, log=True)
    weight_decay = trial.suggest_float("weight_decay", *HPO_WD_RANGE, log=True)
    batch_size = trial.suggest_categorical("batch_size", HPO_BATCH_SIZES)
    scheduler_kwargs = suggest_scheduler_kwargs(trial, scheduler_name)

    full_aug, full_plain = split["full_aug"], split["full_plain"]
    cv_idx, train_labels = split["cv_idx"], split["train_labels"]
    rank_lookup, num_classes = split["rank_lookup"], split["num_classes"]
    pin_memory = (device.type == "cuda")

    if HPO_FOLDS == 1:
        # StratifiedKFold exige n_splits >= 2; com HPO_FOLDS=1 usamos um unico
        # split treino/validacao (estratificado) ao inves de k-fold.
        all_pos = np.arange(len(train_labels))
        train_pos_single, val_pos_single = train_test_split(
            all_pos, test_size=TEST_SIZE, stratify=train_labels, random_state=seed)
        cv_splits = [(train_pos_single, val_pos_single)]
    else:
        skf = StratifiedKFold(n_splits=HPO_FOLDS, shuffle=True, random_state=seed)
        cv_splits = list(skf.split(np.zeros(len(train_labels)), train_labels))

    fold_scores = []

    for fold_i, (train_pos, val_pos) in enumerate(cv_splits):
        train_idx = cv_idx[train_pos]
        val_idx   = cv_idx[val_pos]
        fold_seed = seed * 1000 + fold_i + 1
        seed_everything(fold_seed)
        fold_gen = make_loader_generator(fold_seed)

        train_loader = DataLoader(Subset(full_aug, train_idx), batch_size=batch_size,
                                  num_workers=NUM_WORKERS, pin_memory=pin_memory,
                                  drop_last=True, shuffle=True,
                                  worker_init_fn=seed_worker, generator=fold_gen)
        val_loader = DataLoader(Subset(full_plain, val_idx), batch_size=batch_size,
                                num_workers=NUM_WORKERS, pin_memory=pin_memory, shuffle=False)

        model = build_model(num_classes).to(device)
        fold_train_labels = train_labels[train_pos]
        cw = compute_class_weight("balanced", classes=np.unique(fold_train_labels),
                                  y=fold_train_labels)
        criterion = nn.CrossEntropyLoss(
            weight=torch.tensor(cw, dtype=torch.float32, device=device))
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        scheduler, step_mode = make_scheduler(scheduler_name, optimizer, len(train_loader),
                                              epochs=HPO_TRIAL_EPOCHS, **scheduler_kwargs)

        best_fold_score = -np.inf
        for epoch in range(HPO_TRIAL_EPOCHS):
            model.train()
            for images, labels in train_loader:
                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                outputs = model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                if step_mode == "batch":
                    scheduler.step()

            model.eval()
            v_labels, v_preds = [], []
            val_loss_total, val_total = 0.0, 0
            with torch.no_grad():
                for images, labels in val_loader:
                    images = images.to(device, non_blocking=True)
                    labels = labels.to(device, non_blocking=True)
                    outputs = model(images)
                    val_loss_total += criterion(outputs, labels).item() * labels.size(0)
                    val_total += labels.size(0)
                    v_labels.extend(labels.cpu().numpy())
                    v_preds.extend(torch.argmax(outputs, 1).cpu().numpy())
            avg_val_loss = val_loss_total / val_total

            if step_mode == "epoch":
                scheduler.step()
            elif step_mode == "plateau":
                scheduler.step(avg_val_loss)

            clin = clinical_metrics(v_labels, v_preds, rank_lookup)
            score = composite_score(clin)
            best_fold_score = max(best_fold_score, score)

            if fold_i == 0:
                trial.report(score, epoch)
                if trial.should_prune():
                    del model, optimizer, scheduler
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                    raise optuna.exceptions.TrialPruned()

        fold_scores.append(best_fold_score)
        del model, optimizer, scheduler
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return float(np.mean(fold_scores))

In [10]:
def optimize_config(scheduler_name: str, seed: int, split: dict) -> dict:
    """
    Roda um study Optuna (TPE + HyperbandPruner) por (scheduler, seed) e
    retorna os hiperparametros vencedores, prontos para run_config().
    """
    sampler = optuna.samplers.TPESampler(seed=seed)
    pruner = optuna.pruners.HyperbandPruner(
        min_resource=3, max_resource=HPO_TRIAL_EPOCHS, reduction_factor=3)
    study = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner)

    study.optimize(
        lambda trial: hpo_objective(trial, scheduler_name, seed, split),
        n_trials=HPO_N_TRIALS,
    )

    trials_csv = os.path.join(HPO_TRIALS_DIR, f"trials_{scheduler_name}_seed{seed}.csv")
    study.trials_dataframe().to_csv(trials_csv, index=False)

    best = study.best_trial
    scheduler_kwargs = {k: v for k, v in best.params.items()
                        if k not in ("lr", "weight_decay", "batch_size")}
    return {
        "params": {
            "lr": best.params["lr"],
            "weight_decay": best.params["weight_decay"],
            "batch_size": best.params["batch_size"],
        },
        "scheduler_kwargs": scheduler_kwargs,
        "best_value": best.value,
        "trials_csv": trials_csv,
    }

In [11]:
def run_config(scheduler_name: str, seed: int, split: dict,
               lr: float = None, weight_decay: float = None, batch_size: int = None,
               scheduler_kwargs: dict = None):
    lr = LR if lr is None else lr
    weight_decay = WEIGHT_DECAY if weight_decay is None else weight_decay
    batch_size = BATCH_SIZE if batch_size is None else batch_size
    scheduler_kwargs = scheduler_kwargs or {}

    print("\n" + "#" * 70)
    print(f"# SCHEDULER={scheduler_name}  |  SEED={seed}  |  "
          f"LR={lr:.2e}  WD={weight_decay:.2e}  BS={batch_size}  "
          f"extra={scheduler_kwargs}")
    print("#" * 70)

    seed_everything(seed)
    pin_memory = (device.type == "cuda")
    print(f"Checkpoint metric: {CHECKPOINT_METRIC}")

    full_aug, full_plain = split["full_aug"], split["full_plain"]
    num_classes, class_names = split["num_classes"], split["class_names"]
    rank_lookup, all_labels = split["rank_lookup"], split["all_labels"]
    cv_idx, test_idx, train_labels = split["cv_idx"], split["test_idx"], split["train_labels"]

    classes_idx, classes_count = np.unique(all_labels, return_counts=True)
    print(f"\n--- Dataset: {len(all_labels)} imagens totais, {num_classes} classes ---")
    for c, cnt in zip(classes_idx, classes_count):
        print(f"    {class_names[c]:<12}: {cnt} imagens")

    test_data = Subset(full_plain, test_idx)
    test_loader = DataLoader(test_data, batch_size=batch_size, num_workers=NUM_WORKERS,
                             pin_memory=pin_memory, shuffle=False)

    print(f"--- Split holdout (test_size={TEST_SIZE}): "
          f"treino/CV={len(cv_idx)} imagens | teste={len(test_idx)} imagens ---")

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    cv_metrics = {"f1": [], "auc": [], "score": []}
    ckpt_paths = []

    for fold, (train_pos, val_pos) in enumerate(
            skf.split(np.zeros(len(train_labels)), train_labels)):
        print(f"\n === Fold {fold + 1}/{N_FOLDS} ({scheduler_name}, seed {seed}) ===")
        train_idx = cv_idx[train_pos]
        val_idx   = cv_idx[val_pos]
        print(f"     treino={len(train_idx)} imagens | validacao={len(val_idx)} imagens")
        train_subset = Subset(full_aug,   train_idx)
        val_subset   = Subset(full_plain, val_idx)
        fold_seed = seed * 1000 + fold + 1
        seed_everything(fold_seed)
        fold_gen = make_loader_generator(fold_seed)

        train_loader = DataLoader(train_subset, batch_size=batch_size, num_workers=NUM_WORKERS,
                                  pin_memory=pin_memory, drop_last=True, shuffle=True,
                                  worker_init_fn=seed_worker, generator=fold_gen)
        val_loader   = DataLoader(val_subset, batch_size=batch_size, num_workers=NUM_WORKERS,
                                  pin_memory=pin_memory, shuffle=False)

        model = build_model(num_classes).to(device)

        fold_train_labels = train_labels[train_pos]
        cw = compute_class_weight("balanced",
                                  classes=np.unique(fold_train_labels),
                                  y=fold_train_labels)
        criterion = nn.CrossEntropyLoss(
            weight=torch.tensor(cw, dtype=torch.float32, device=device))
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

        steps_per_epoch = len(train_loader)
        scheduler, step_mode = make_scheduler(scheduler_name, optimizer, steps_per_epoch,
                                              epochs=EPOCHS, **scheduler_kwargs)

        best_score, best_val_auc, best_val_f1 = -np.inf, 0.0, 0.0
        best_val_clin = None
        history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "lrs": []}
        ckpt_path = os.path.join(
            OUTPUT_DIR, f"{scheduler_name}_seed{seed}_fold{fold + 1}_best.pth")

        for epoch in range(EPOCHS):
            model.train()
            train_loss, train_correct, train_total = 0.0, 0, 0
            for images, labels in train_loader:
                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                outputs = model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

                batch_sz = labels.size(0)
                train_loss += loss.item() * batch_sz
                train_correct += (torch.argmax(outputs, 1) == labels).sum().item()
                train_total += batch_sz
                if step_mode == "batch":
                    scheduler.step()
            avg_train_loss = train_loss / train_total
            train_acc = train_correct / train_total

            model.eval()
            val_loss, val_total = 0.0, 0
            v_labels, v_preds, v_probs = [], [], []
            with torch.no_grad():
                for images, labels in val_loader:
                    images = images.to(device, non_blocking=True)
                    labels = labels.to(device, non_blocking=True)
                    outputs = model(images)
                    batch_sz = labels.size(0)
                    val_loss += criterion(outputs, labels).item() * batch_sz
                    val_total += batch_sz
                    v_labels.extend(labels.cpu().numpy())
                    v_preds.extend(torch.argmax(outputs, 1).cpu().numpy())
                    v_probs.extend(F.softmax(outputs, 1).cpu().numpy())

            avg_val_loss = val_loss / val_total
            val_f1 = f1_score(v_labels, v_preds, average="macro")
            val_acc = accuracy_score(v_labels, v_preds)
            try:
                val_auc = roc_auc_score(v_labels, v_probs,
                                        multi_class="ovr", average="macro")
            except ValueError:
                val_auc = float("nan")
            selection_score, val_clin = checkpoint_selection_score(
                val_auc, val_f1, v_labels, v_preds, rank_lookup
            )

            if step_mode == "epoch":
                scheduler.step()
            elif step_mode == "plateau":
                scheduler.step(avg_val_loss)
            current_lr = optimizer.param_groups[0]['lr']

            history["train_loss"].append(avg_train_loss)
            history["val_loss"].append(avg_val_loss)
            history["train_acc"].append(train_acc)
            history["val_acc"].append(val_acc)
            history["lrs"].append(current_lr)

            print(f"Ep {epoch + 1:02d}/{EPOCHS} | "
                  f"LR {current_lr:.2e} | "
                  f"TrainLoss {avg_train_loss:.4f} | TrainAcc {train_acc:.4f} | "
                  f"ValLoss {avg_val_loss:.4f} | ValAcc {val_acc:.4f} | "
                  f"F1 {val_f1:.4f} | AUC {val_auc:.4f} | "
                  f"QWK {val_clin['qwk']:.4f} | Score {selection_score:.4f}")

            if selection_score > best_score:
                best_score, best_val_auc, best_val_f1 = selection_score, val_auc, val_f1
                best_val_clin = val_clin
                torch.save(model.state_dict(), ckpt_path)

        if not os.path.isfile(ckpt_path):
            warnings.warn(f"Nenhum checkpoint selecionado em {ckpt_path}; salvando estado final do fold.")
            torch.save(model.state_dict(), ckpt_path)
        plot_fold_telemetry(history, fold + 1, scheduler_name, seed)
        print(f"     melhor checkpoint: score={best_score:.4f} | "
              f"AUC={best_val_auc:.4f} | F1={best_val_f1:.4f}")

        cv_metrics["auc"].append(best_val_auc)
        cv_metrics["f1"].append(best_val_f1)
        cv_metrics["score"].append(best_score)
        ckpt_paths.append(ckpt_path)

    auc_mean, auc_std = np.mean(cv_metrics["auc"]), np.std(cv_metrics["auc"])
    f1_mean,  f1_std  = np.mean(cv_metrics["f1"]),  np.std(cv_metrics["f1"])
    score_mean, score_std = np.mean(cv_metrics["score"]), np.std(cv_metrics["score"])
    print(f"\n--- CV {scheduler_name}/seed{seed}: "
          f"AUC {auc_mean:.4f}±{auc_std:.4f} | "
          f"F1 {f1_mean:.4f}±{f1_std:.4f} | "
          f"Score {score_mean:.4f}±{score_std:.4f} ---")

    ensemble = []
    for p in ckpt_paths:
        m = build_model(num_classes).to(device)
        m.load_state_dict(torch.load(p, map_location=device, weights_only=True))
        m.eval()
        ensemble.append(m)

    test_labels, test_preds, test_probs = [], [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device, non_blocking=True)
            batch_probs = torch.zeros((images.size(0), num_classes), device=device)
            for m in ensemble:
                batch_probs += F.softmax(m(images), dim=1)
            batch_probs /= N_FOLDS
            test_labels.extend(labels.numpy())
            test_preds.extend(torch.argmax(batch_probs, 1).cpu().numpy())
            test_probs.extend(batch_probs.cpu().numpy())

    test_f1  = f1_score(test_labels, test_preds, average="macro")
    test_auc = roc_auc_score(test_labels, test_probs, multi_class="ovr", average="macro")
    cm_test  = confusion_matrix(test_labels, test_preds, labels=range(num_classes))
    clin = clinical_metrics(test_labels, test_preds, rank_lookup)

    print(f">>> TEST {scheduler_name}/seed{seed}: F1 {test_f1:.4f} | "
          f"AUC {test_auc:.4f} | QWK {clin['qwk']:.4f} | "
          f"Erros>=2 {clin['danger_count']} ({clin['danger_rate']*100:.1f}%)")

    return {
        "Timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "Modelo": MODEL_NAME,
        "Scheduler": scheduler_name,
        "Seed": seed,
        "Folds": N_FOLDS, "Epochs": EPOCHS, "Batch_Size": batch_size,
        "LR": lr, "Weight_Decay": weight_decay, "Optimizer": "AdamW",
        "Checkpoint_Metric": CHECKPOINT_METRIC,
        "Classes": str(class_names), "Test_Size": TEST_SIZE,
        "Val_CV_AUC_Mean": f"{auc_mean:.4f}", "Val_CV_AUC_Std": f"{auc_std:.4f}",
        "Val_CV_F1_Mean":  f"{f1_mean:.4f}",  "Val_CV_F1_Std":  f"{f1_std:.4f}",
        "Val_CV_Score_Mean": f"{score_mean:.4f}", "Val_CV_Score_Std": f"{score_std:.4f}",
        "Test_Ensemble_AUC": f"{test_auc:.4f}", "Test_Ensemble_F1": f"{test_f1:.4f}",
        "Test_QWK": f"{clin['qwk']:.4f}", "Test_Accuracy": f"{clin['acc']:.4f}",
        "Test_Danger_Count": clin["danger_count"],
        "Test_Danger_Rate": f"{clin['danger_rate']:.4f}",
        "Test_Underest_Count": clin["underest_count"],
        "Test_Underest_Rate": f"{clin['underest_rate']:.4f}",
        "Test_Confusion_Matrix": str(cm_test.tolist()),
    }

In [12]:
# =============================================================================
# 4. LOGGING  (colunas HPO_* extras em relacao ao pipeline_resnet18.ipynb)
# =============================================================================
FIELDNAMES = [
    "Timestamp", "Modelo", "Scheduler", "Seed", "Folds", "Epochs", "Batch_Size",
    "LR", "Weight_Decay", "Optimizer", "Checkpoint_Metric", "Classes", "Test_Size",
    "Val_CV_AUC_Mean", "Val_CV_AUC_Std", "Val_CV_F1_Mean", "Val_CV_F1_Std",
    "Val_CV_Score_Mean", "Val_CV_Score_Std",
    "Test_Ensemble_AUC", "Test_Ensemble_F1", "Test_QWK", "Test_Accuracy",
    "Test_Danger_Count", "Test_Danger_Rate", "Test_Underest_Count",
    "Test_Underest_Rate", "Test_Confusion_Matrix",
    "HPO_Best_Value", "HPO_N_Trials", "HPO_Trial_Epochs", "HPO_Folds",
    "HPO_Scheduler_Params", "HPO_Trials_CSV",
]


def log_row(row: dict):
    exists = os.path.isfile(CSV_FILE)
    if exists:
        with open(CSV_FILE, newline="", encoding="utf-8") as f:
            header = next(csv.reader(f), [])
        if header != FIELDNAMES:
            raise ValueError(
                f"CSV existente tem colunas incompativeis com o pipeline atual: {CSV_FILE}"
            )
    with open(CSV_FILE, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
        if not exists:
            writer.writeheader()
        writer.writerow(row)


def get_completed_configs(csv_path: str = CSV_FILE) -> set:
    """Le o CSV (se existir) e retorna (Scheduler, Seed) ja concluidos, para resume."""
    if not os.path.isfile(csv_path):
        return set()
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        return {(row["Scheduler"], int(row["Seed"])) for row in reader}


# =============================================================================
# 5. RESUMO ESTATISTICO (le o CSV; nao re-treina nem re-otimiza)
# =============================================================================
import ast


def _expand_cm(cm):
    """Matriz de confusao -> (y_true, y_pred) repetindo cada celula."""
    yt, yp = [], []
    for i, row in enumerate(cm):
        for j, c in enumerate(row):
            yt += [i] * int(c)
            yp += [j] * int(c)
    return np.array(yt), np.array(yp)

In [13]:
def _ensure_metric_cols(df):
    """
    Garante colunas Test_QWK / Test_Danger_Rate / Test_Accuracy.
    Se faltarem (ex.: CSVs antigos), recalcula a partir de Test_Confusion_Matrix
    usando a ordem ORDINAL clinica.
    """
    need = {"Test_QWK", "Test_Danger_Rate", "Test_Accuracy"} - set(df.columns)
    if not need:
        return df
    qwks, drates, accs = [], [], []
    for _, r in df.iterrows():
        try:
            class_names = ast.literal_eval(r["Classes"])
        except Exception:
            class_names = ORDINAL_ORDER
        rank_lookup = ordinal_ranks(class_names)
        cm = np.array(ast.literal_eval(r["Test_Confusion_Matrix"]))
        yt, yp = _expand_cm(cm.tolist())
        m = clinical_metrics(yt, yp, rank_lookup)
        qwks.append(m["qwk"]); drates.append(m["danger_rate"]); accs.append(m["acc"])
    if "Test_QWK" not in df.columns:          df["Test_QWK"] = qwks
    if "Test_Danger_Rate" not in df.columns:  df["Test_Danger_Rate"] = drates
    if "Test_Accuracy" not in df.columns:     df["Test_Accuracy"] = accs
    return df

In [14]:
def summarize_ablation(csv_path: str = CSV_FILE):
    import pandas as pd
    try:
        from scipy import stats
        have_scipy = True
    except ImportError:
        have_scipy = False

    if not os.path.isfile(csv_path):
        print(f"[aviso] CSV nao encontrado: {csv_path}")
        print("        Rode main(run_training=True) para gerar os resultados desta configuracao.")
        return

    df = pd.read_csv(csv_path)
    df = _ensure_metric_cols(df)
    for col in ["Test_QWK", "Test_Ensemble_F1", "Test_Ensemble_AUC",
                "Test_Danger_Rate", "Test_Accuracy"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    metrics = [("Test_QWK", "QWK", "max"),
               ("Test_Ensemble_F1", "F1", "max"),
               ("Test_Ensemble_AUC", "AUC", "max"),
               ("Test_Accuracy", "Acc", "max"),
               ("Test_Danger_Rate", "Erro>=2", "min")]

    print("\n" + "=" * 78)
    print(" RESUMO POR SCHEDULER  (media +/- desvio entre seeds, cada uma com seu proprio HPO)")
    print("=" * 78)
    order = [s for s in SCHEDULERS if s in df["Scheduler"].unique()]
    extra = [s for s in df["Scheduler"].unique() if s not in order]
    order += extra
    head = f"{'Scheduler':<20}" + "".join(f"{lbl:<16}" for _, lbl, _ in metrics) + "n"
    print(head)
    print("-" * len(head))
    for sch in order:
        g = df[df["Scheduler"] == sch]
        cells = ""
        for col, _, _ in metrics:
            cells += f"{g[col].mean():.4f}+/-{g[col].std(ddof=1):.3f}  "
        print(f"{sch:<20}{cells}{len(g)}")

    print("\nMelhor media por metrica:")
    for col, lbl, direction in metrics:
        means = df.groupby("Scheduler")[col].mean()
        best = means.idxmin() if direction == "min" else means.idxmax()
        print(f"  {lbl:<10} -> {best}  ({means[best]:.4f})")

    if not have_scipy:
        print("\n[aviso] scipy nao instalado: pulei os testes de significancia.")
        print("        instale com: pip install scipy")
        return

    print("\n" + "=" * 78)
    print(" TESTES DE SIGNIFICANCIA PAREADOS (pareados por seed)")
    print("=" * 78)
    n_seeds = df["Seed"].nunique()
    print(f"n_seeds={n_seeds}.", end=" ")
    if n_seeds < 6:
        print("ATENCAO: com n<6, o Wilcoxon nao alcanca p<0.05 mesmo no caso\n"
              "         extremo (p minimo bilateral = 0.0625). Trate como exploratorio;\n"
              "         para conclusao formal, aumente o numero de seeds.")
    else:
        print("")

    for col, lbl, direction in metrics:
        piv = df.pivot_table(index="Seed", columns="Scheduler",
                             values=col, aggfunc="mean")
        schs = [s for s in order if s in piv.columns]
        sub = piv[schs].dropna(axis=0, how="any")
        if len(sub) < 2 or len(schs) < 2:
            print(f"\n[{lbl}] dados insuficientes para teste pareado.")
            continue

        print(f"\n[{lbl}]  (seeds pareadas usadas: {len(sub)})")
        if len(schs) >= 3:
            try:
                chi2, p = stats.friedmanchisquare(*[sub[s].values for s in schs])
                print(f"  Friedman (omnibus): chi2={chi2:.3f}  p={p:.4f}")
            except Exception as e:
                print(f"  Friedman: nao aplicavel ({e})")
        better = max if direction == "max" else min
        for a in range(len(schs)):
            for b in range(a + 1, len(schs)):
                xa, xb = sub[schs[a]].values, sub[schs[b]].values
                diff = xa - xb
                try:
                    w_p = stats.wilcoxon(xa, xb, zero_method="wilcox").pvalue
                except ValueError:
                    w_p = float("nan")
                t_p = stats.ttest_rel(xa, xb).pvalue
                win = schs[a] if better(xa.mean(), xb.mean()) == xa.mean() else schs[b]
                print(f"    {schs[a]:<18} vs {schs[b]:<18} | "
                      f"d_medio={diff.mean():+.4f} | Wilcoxon p={w_p:.4f} | "
                      f"t-pareado p={t_p:.4f} | tende a favor de: {win}")

    print("\n[nota] QWK/F1/AUC/Acc: maior e' melhor. Erro>=2: menor e' melhor.")
    print("[nota] Diferente de pipeline_resnet18.ipynb, aqui cada scheduler tambem")
    print("       recebeu seu proprio HPO -- ver colunas HPO_* para os hiperparametros")
    print("       vencedores de cada combinacao.")
    print("=" * 78)

In [ ]:
def main(run_training: bool = True):
    if run_training:
        completed = get_completed_configs(CSV_FILE)
        if completed:
            print(f"[resume] {len(completed)} combinacoes ja concluidas em {CSV_FILE}; serao puladas.")

        for scheduler_name in SCHEDULERS:
            for seed in SEEDS:
                if (scheduler_name, seed) in completed:
                    print(f"[SKIP] {scheduler_name}/seed{seed} ja esta em {CSV_FILE}")
                    continue

                split = prepare_split(seed)

                print("\n" + "=" * 70)
                print(f"[HPO] {scheduler_name}/seed{seed}: {HPO_N_TRIALS} trials x "
                      f"{HPO_TRIAL_EPOCHS} epocas x {HPO_FOLDS} fold(s)")
                print("=" * 70)
                hpo_result = optimize_config(scheduler_name, seed, split)
                print(f"[HPO] Melhor config ({scheduler_name}/seed{seed}): "
                      f"{hpo_result['params']} | extra={hpo_result['scheduler_kwargs']} | "
                      f"score={hpo_result['best_value']:.4f}")

                row = run_config(
                    scheduler_name, seed, split,
                    lr=hpo_result["params"]["lr"],
                    weight_decay=hpo_result["params"]["weight_decay"],
                    batch_size=hpo_result["params"]["batch_size"],
                    scheduler_kwargs=hpo_result["scheduler_kwargs"],
                )
                row["HPO_Best_Value"] = f"{hpo_result['best_value']:.4f}"
                row["HPO_N_Trials"] = HPO_N_TRIALS
                row["HPO_Trial_Epochs"] = HPO_TRIAL_EPOCHS
                row["HPO_Folds"] = HPO_FOLDS
                row["HPO_Scheduler_Params"] = str(hpo_result["scheduler_kwargs"])
                row["HPO_Trials_CSV"] = hpo_result["trials_csv"]

                log_row(row)
                completed.add((scheduler_name, seed))
                print(f"[LOG] {scheduler_name}/seed{seed} salvo em {CSV_FILE}")

        print("\n" + "=" * 70)
        print(f"Ablacao com HPO concluida: {len(SCHEDULERS) * len(SEEDS)} experimentos.")
        print(f"Resultados em: {CSV_FILE}")
        print("=" * 70)

    summarize_ablation(CSV_FILE)


if __name__ == "__main__":
    import sys
    if len(sys.argv) > 1 and sys.argv[1] == "--summary":
        path = sys.argv[2] if len(sys.argv) > 2 else CSV_FILE
        summarize_ablation(path)
        sys.exit(0)
    main()

[resume] 21 combinacoes ja concluidas em ./ablation_schedulers_resnet50_optuna_ckpt_clinical_composite\ablation_logs_resnet50_optuna.csv; serao puladas.
[SKIP] CosineAnnealingLR/seed42 ja esta em ./ablation_schedulers_resnet50_optuna_ckpt_clinical_composite\ablation_logs_resnet50_optuna.csv
[SKIP] CosineAnnealingLR/seed17 ja esta em ./ablation_schedulers_resnet50_optuna_ckpt_clinical_composite\ablation_logs_resnet50_optuna.csv
[SKIP] CosineAnnealingLR/seed7 ja esta em ./ablation_schedulers_resnet50_optuna_ckpt_clinical_composite\ablation_logs_resnet50_optuna.csv
[SKIP] CosineAnnealingLR/seed12 ja esta em ./ablation_schedulers_resnet50_optuna_ckpt_clinical_composite\ablation_logs_resnet50_optuna.csv
[SKIP] CosineAnnealingLR/seed123 ja esta em ./ablation_schedulers_resnet50_optuna_ckpt_clinical_composite\ablation_logs_resnet50_optuna.csv
[SKIP] CosineAnnealingLR/seed555 ja esta em ./ablation_schedulers_resnet50_optuna_ckpt_clinical_composite\ablation_logs_resnet50_optuna.csv
[SKIP] Cosin

: 